# unify_pums.ipynb

This notebook reads in the household and person-level PUMS for a given year, merges them, cleans up the ORIGIN/CHOSEN fields, and injects fields that will be used later on during modeling.

In [1]:
import os
import sys

import numpy as np
import pandas as pd

sys.path.insert(0, os.path.abspath("../.."))


In [2]:
sample_size = 1  # proportion of people in PUMS to consider
year = 2018
path = "us/pums_2018_raw.csv"

In [3]:
# Read only the columns this notebook actually uses -- 33 of the 300 in the IPUMS
# extract.
#
# This list is exhaustive for the notebook as written. Adding a variable downstream
# means adding it here too, otherwise it will KeyError rather than silently go missing.
USECOLS = [
    # identity / weight
    "CBSERIAL",
    "PERNUM",
    "PERWT",
    # geography and the origin/destination outcome
    "STATEFIP",
    "PUMA",
    "MIGPLAC1",
    "MIGPUMA1",
    "MIGPUMANOW",
    # decision-unit construction
    "SUBFAM",
    "SFRELATE",
    "RELATE",
    "RELATED",
    "AGE",
    # per-member covariates (SHARED_COLS)
    "GRADEATT",
    "RACE",
    "HISPAN",
    "BPL",
    "CITIZEN",
    "INDNAICS",
    "CLASSWKR",
    "MARST",
    "DIVINYR",
    "WIDINYR",
    "MARRINYR",
    "VETSTATD",
    "EMPSTAT",
    "EDUC",
    "EDUCD",
    "RACAMIND",
    "RACASIAN",
    "RACBLK",
    "RACPACIS",
    "RACWHT",
    "RACOTHER",
]


df = pd.read_csv(path, usecols=USECOLS)
df

,CBSERIAL,STATEFIP,PUMA,PERNUM,PERWT,SUBFAM,SFRELATE,RELATE,RELATED,AGE,...,EDUC,EDUCD,GRADEATT,EMPSTAT,CLASSWKR,INDNAICS,MIGPLAC1,MIGPUMA1,MIGPUMANOW,VETSTATD
0,2018010000049,1,1600,1,75.0,0,0,12,1270,19,...,6,65,6,3,2,44511,13,3400,1590,11
1,2018010000058,1,1900,1,75.0,0,0,12,1270,18,...,6,65,6,3,2,722Z,13,1900,1900,11
2,2018010000219,1,2000,1,118.0,0,0,13,1301,53,...,6,64,0,3,2,9211MP,0,0,2090,11
3,2018010000246,1,2400,1,43.0,0,0,13,1301,28,...,7,71,0,3,0,0,0,0,2400,20
4,2018010000251,1,2701,1,16.0,0,0,13,1301,25,...,3,30,0,3,2,333MS,1,2700,2700,11
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3214534,2018001400326,56,400,4,87.0,0,0,12,1260,9,...,1,16,3,0,0,0,0,0,400,0
3214535,2018001400326,56,400,5,87.0,0,0,12,1260,7,...,1,12,3,0,0,0,0,0,400,0
3214536,2018001400502,56,100,1,49.0,0,0,1,101,49,...,7,71,0,1,2,6231,0,0,100,11
3214537,2018001400502,56,100,2,46.0,0,0,3,301,19,...,6,63,0,1,2,115,0,0,100,11


In [4]:
df["PID"] = df["CBSERIAL"] * 1_000 + df["PERNUM"]
assert df["PID"].is_unique
df = df.set_index("PID")

In [5]:
df["MIGPLAC1"].value_counts()

MIGPLAC1
0      2788765
6        46187
48       36447
12       27232
36       20896
        ...   
330         75
350         73
520         71
623         66
622         61
Name: count, Length: 108, dtype: int64

In [6]:
df["PUMA"]

PID
2018010000049001    1600
2018010000058001    1900
2018010000219001    2000
2018010000246001    2400
2018010000251001    2701
                    ... 
2018001400326004     400
2018001400326005     400
2018001400502001     100
2018001400502002     100
2018001400515001     500
Name: PUMA, Length: 3214539, dtype: int64

In [7]:
# origin is the MIGSP + MIGPUMA
# need ints since it is treated as a float by default
df["ORIGIN"] = df["MIGPLAC1"].astype(int).astype(str).str.zfill(2) + df[
    "MIGPUMA1"
].astype(int).astype(str).str.zfill(5)  # migpuma geography

# chosen is the current location, ST + PUMA
df["CHOSEN"] = df["STATEFIP"].astype(int).astype(str).str.zfill(2) + df["PUMA"].astype(
    int
).astype(str).str.zfill(5)  # puma geography

df["CHOSEN_MIGPUMA"] = df["STATEFIP"].astype(int).astype(str).str.zfill(2) + df[
    "MIGPUMANOW"
].astype(int).astype(str).str.zfill(5)

In [8]:
df["ORIGIN"].value_counts()

ORIGIN
0000000     2788765
0603700       10810
0400100        5848
1703400        5806
2500390        5542
             ...   
33000001         75
35000001         73
52000001         71
62300001         66
62200001         61
Name: count, Length: 1039, dtype: int64

In [9]:
# backfill the stay origins to the MIGPUMA where they are currently (chosen == origin)
df["ORIGIN"] = np.where(
    df["ORIGIN"] == "0000000",
    df["CHOSEN_MIGPUMA"],
    df["ORIGIN"],
)
# fill in the origin state with this backfill in place
df["ORIGIN_STATE"] = np.where(
    df["ORIGIN"].str.len() == 7, df["ORIGIN"].str[:2].astype(int), 999
)

# define STAY as moving outside the MIGPUMA
df["STAY"] = np.where(df["CHOSEN_MIGPUMA"] == df["ORIGIN"], 1, 0)

In [10]:
df["ORIGIN"].value_counts()

ORIGIN
0603700     102203
2500390      49638
1703400      41902
0400100      41174
4804600      36741
             ...  
33000001        75
35000001        73
52000001        71
62300001        66
62200001        61
Name: count, Length: 1038, dtype: int64

In [11]:
df["ORIGIN_STATE"].value_counts()

ORIGIN_STATE
6      377274
48     265623
12     199113
36     197113
42     128757
17     126968
39     118907
37     101294
13      99777
26      99181
34      88714
51      84057
53      75492
25      69599
4       68702
18      67427
47      67323
29      62272
24      59648
55      59634
27      55797
8       55563
45      49125
1       47512
21      45229
22      43570
41      41714
40      37582
9       36349
19      32195
49      31177
5       30415
20      29519
28      29075
32      28499
31      19450
35      19155
54      18106
999     17841
16      16517
15      14390
33      13648
23      13117
44      10325
30      10287
46       9075
10       9056
38       7882
2        6858
11       6510
50       6388
56       5738
Name: count, dtype: int64

In [12]:
df["CHOSEN"].value_counts()

CHOSEN
0102500    4550
5310200    4083
5500700    4082
5500100    3958
1200500    3421
           ... 
2701503     580
5541001     575
2701403     566
2701402     523
4203207     509
Name: count, Length: 2351, dtype: int64

In [13]:
df["STAY"].value_counts()

STAY
1    3025964
0     188575
Name: count, dtype: int64

In [14]:
for c in df.columns:
    print(c)

CBSERIAL
STATEFIP
PUMA
PERNUM
PERWT
SUBFAM
SFRELATE
RELATE
RELATED
AGE
MARST
MARRINYR
DIVINYR
WIDINYR
RACE
HISPAN
BPL
CITIZEN
RACAMIND
RACASIAN
RACBLK
RACPACIS
RACWHT
RACOTHER
EDUC
EDUCD
GRADEATT
EMPSTAT
CLASSWKR
INDNAICS
MIGPLAC1
MIGPUMA1
MIGPUMANOW
VETSTATD
ORIGIN
CHOSEN
CHOSEN_MIGPUMA
ORIGIN_STATE
STAY


In [15]:
df["IS_CHILD_UNDER_6"] = np.where(df["AGE"] < 6, 1, 0)
df["IS_CHILD_6_TO_17"] = np.where((df["AGE"] >= 6) & (df["AGE"] <= 17), 1, 0)
df["IS_65_OR_OLDER"] = np.where(df["AGE"] >= 65, 1, 0)
df["IN_LF"] = np.where(df["EMPSTAT"].isin([1, 2]), 1, 0)
df["WORKING"] = np.where(df["EMPSTAT"] == 1, 1, 0)

In [16]:
PRIMARY = {1, 2}
df["UNIT"] = np.where(
    # sometimes subfam != 0 while sfrelate == 0
    (df.SUBFAM != 0) & (df.SFRELATE != 0),
    # subfamily → its own unit
    df.CBSERIAL.astype(str) + "_SF" + df.SUBFAM.astype(str),
    np.where(
        # count primary
        df.RELATE.isin(PRIMARY)
        # count related children don't count unrelated children, count ofster children
        | ((df.AGE < 18) & ((df.RELATE < 11) | (df.RELATED.isin([1242]))))
        # unmarried partner
        | df.RELATED.isin([1114]),
        # primary family, non-subfamily or child with age < 18
        df.CBSERIAL.astype(str) + "_P",
        # treat everyone else as singletons, individual decision units
        df.CBSERIAL.astype(str) + "_I" + df.PERNUM.astype(int).astype(str),
    ),
)

In [17]:
df["_REF_PRIORITY"] = np.where((df["SFRELATE"] == 1) | (df["RELATE"] == 1), 0, 1)
df["_SEC_PRIORITY"] = np.where(
    (df["SFRELATE"] == 2) | (df["RELATE"] == 2) | (df["RELATED"] == 1114), 0, 1
)

units = df.groupby("UNIT").agg(
    REF_INDEX=("_REF_PRIORITY", "idxmin"),
    SEC_INDEX=("_SEC_PRIORITY", "idxmin"),
    NUM_CHILDREN_UNDER_6=("IS_CHILD_UNDER_6", "sum"),
    NUM_CHILDREN_6_TO_17=("IS_CHILD_6_TO_17", "sum"),
    SIZE=("_REF_PRIORITY", "size"),
    NUM_IN_IF=("IN_LF", "sum"),
    NUM_WORKING=("WORKING", "sum"),
)

has_sec = df.groupby("UNIT")["_SEC_PRIORITY"].min().eq(0)
units["SEC_INDEX"] = units["SEC_INDEX"].where(has_sec, units["REF_INDEX"])

In [18]:
CATEGORIES = ["REF", "SEC"]

In [19]:
SHARED_COLS = [
    "GRADEATT",
    "RACE",
    "HISPAN",
    "BPL",
    "CITIZEN",
    "WORKING",
    "INDNAICS",
    "CLASSWKR",
    "AGE",
    "MARST",
    "DIVINYR",
    "WIDINYR",
    "MARRINYR",
    "VETSTATD",
    "EMPSTAT",
    "RACAMIND",
    "RACASIAN",
    "RACBLK",
    "RACPACIS",
    "RACWHT",
    "RACOTHER",
    "EDUC",
    "EDUCD",
    "RELATE",
]
BASE_COLS = ["PERWT", "CHOSEN", "ORIGIN", "STAY", "STATEFIP", "ORIGIN_STATE"]

for c in BASE_COLS:
    units[c] = df.loc[units["REF_INDEX"].values, c].values

for c in SHARED_COLS:
    for category in CATEGORIES:
        units[f"{c}_{category}"] = df.loc[units[f"{category}_INDEX"].values, c].values

In [20]:
mask = (
    (units["AGE_REF"] >= 18)
    & (units["AGE_SEC"] >= 18)
    # do not consider institutional inmates
    & (units["RELATE_REF"] != 13)
    & (units["ORIGIN_STATE"] <= 56)
    & (~units["ORIGIN_STATE"].isin([2, 15]))
    & (units["STATEFIP"] <= 56)
    & (~units["STATEFIP"].isin([2, 15]))
)

print(units.shape)
units_subset = units.loc[mask].copy()
print(units_subset.shape)

(1845792, 61)
(1739030, 61)


In [21]:
units_subset["AGE_REF"].value_counts()

AGE_REF
18    44051
19    42003
20    37117
21    34246
60    31266
      ...  
95     3540
93     2540
92     2501
91     1525
96      172
Name: count, Length: 79, dtype: int64

In [22]:
units_sample = units_subset.sample(frac=sample_size, random_state=4703213)
units_sample.shape

(1739030, 61)

In [23]:
units_sample["PAIRED_UNIT"] = np.where(
    units_sample["REF_INDEX"] != units_sample["SEC_INDEX"], 1, 0
)

In [24]:
units_sample["CHILD_UNDER_6"] = np.where(units_sample["NUM_CHILDREN_UNDER_6"] > 0, 1, 0)
units_sample["CHILD_6_TO_17"] = np.where(units_sample["NUM_CHILDREN_6_TO_17"] > 0, 1, 0)
units_sample["CHILD"] = np.where(
    (units_sample["CHILD_UNDER_6"] == 1) | (units_sample["CHILD_6_TO_17"] == 1), 1, 0
)
units_sample["NUM_CHILDREN"] = (
    units_sample["NUM_CHILDREN_UNDER_6"] + units_sample["NUM_CHILDREN_6_TO_17"]
)

In [25]:
# Earner counts. WORK1 means "exactly one earner in the unit" -- for a couple that is
# XOR, for a single-person unit it is simply whether that person works. Previously
# WORK1 was 1 for EVERY unpaired unit regardless of employment (only 54% of which
# actually had a working reference person), and PAIR_WORK1 was identically 1 for all
# rows, so it carried no information at all.
_w_ref = units_sample["WORKING_REF"].astype(bool)
_w_sec = units_sample["WORKING_SEC"].astype(bool)
_paired = units_sample["PAIRED_UNIT"] == 1

units_sample["WORK2"] = np.where(_paired & _w_ref & _w_sec, 1, 0)
units_sample["WORK1"] = np.where(np.where(_paired, _w_ref ^ _w_sec, _w_ref), 1, 0)
# couples with exactly one earner -- the classic tied-mover case
units_sample["PAIR_WORK1"] = np.where(_paired & (units_sample["WORK1"] == 1), 1, 0)

units_sample["SINGLE_UNIT_WITH_CHILD"] = np.where(
    (units_sample["PAIRED_UNIT"] == 0) & (units_sample["CHILD"] == 1), 1, 0
)

In [26]:
# Education brackets. IPUMS EDUC is a coarse 0-11 scale and merges "12th grade, no
# diploma" with high-school graduates, so the brackets are built from EDUCD (detailed,
# 0-116), which maps 1:1 onto the ACS SCHL categories this pipeline used before:
#   <=61  less than a HS diploma (incl. 61 = 12th grade, no diploma)
#   62-64 HS diploma or GED
#   65-90 some college / associate's
#   101   bachelor's
#   >=110 graduate or professional
units_sample["MAX_EDUC"] = units_sample[["EDUC_REF", "EDUC_SEC"]].max(axis=1)
units_sample["MAX_EDUCD"] = units_sample[["EDUCD_REF", "EDUCD_SEC"]].max(axis=1)

_e = units_sample["MAX_EDUCD"]
units_sample["EDU_NOHIGH"] = np.where(_e <= 61, 1, 0)
units_sample["EDU_ONLY_HIGH"] = np.where(_e.isin([62, 63, 64]), 1, 0)
units_sample["EDU_SOME_COLLEGE"] = np.where(_e.isin([65, 70, 71, 80, 81, 90]), 1, 0)
units_sample["EDU_ONLY_BACHELORS"] = np.where(_e == 101, 1, 0)
units_sample["EDU_GRADUATE_DEG"] = np.where(_e >= 110, 1, 0)
units_sample["EDU_BACHELORS_OR_HIGHER"] = np.where(_e >= 101, 1, 0)
units_sample["EDU_HAS_DEGREE"] = np.where(_e >= 101, 1, 0)
units_sample["EDU_NO_DEGREE"] = np.where(_e < 101, 1, 0)
units_sample["EDU_HIGH_BUT_NOT_BACHELORS"] = np.where((_e >= 62) & (_e < 101), 1, 0)

# the five mutually exclusive brackets must partition every unit -- create_estdata
# gathers OWN_EARNINGS_10K_BY_EDU off exactly one of them
_excl = [
    "EDU_NOHIGH",
    "EDU_ONLY_HIGH",
    "EDU_SOME_COLLEGE",
    "EDU_ONLY_BACHELORS",
    "EDU_GRADUATE_DEG",
]
assert (units_sample[_excl].sum(axis=1) == 1).all(), (
    "EDU_* brackets do not partition the units"
)

In [27]:
# MEAN_AGE is the mean of two integers, so it can land on x.5. Upper edges are
# exclusive (< 35, < 65) rather than <= 34 / <= 64, otherwise units at 34.5 and 64.5
# fall into no bracket at all -- 13,919 of them before this fix.
units_sample["MEAN_AGE"] = units_sample[["AGE_REF", "AGE_SEC"]].mean(axis=1)
units_sample["AGE_UNDER_18"] = np.where(units_sample["MEAN_AGE"] < 18, 1, 0)
units_sample["AGE_18_34"] = np.where(
    (units_sample["MEAN_AGE"] >= 18) & (units_sample["MEAN_AGE"] < 35), 1, 0
)
units_sample["AGE_35_64"] = np.where(
    (units_sample["MEAN_AGE"] >= 35) & (units_sample["MEAN_AGE"] < 65), 1, 0
)
units_sample["AGE_OVER_65"] = np.where(units_sample["MEAN_AGE"] >= 65, 1, 0)

units_sample["AGE_18_22"] = np.where(
    (units_sample["MEAN_AGE"] >= 18) & (units_sample["MEAN_AGE"] < 23), 1, 0
)
units_sample["AGE_23_29"] = np.where(
    (units_sample["MEAN_AGE"] >= 23) & (units_sample["MEAN_AGE"] < 30), 1, 0
)
units_sample["AGE_30_39"] = np.where(
    (units_sample["MEAN_AGE"] >= 30) & (units_sample["MEAN_AGE"] < 40), 1, 0
)
units_sample["AGE_40_49"] = np.where(
    (units_sample["MEAN_AGE"] >= 40) & (units_sample["MEAN_AGE"] < 50), 1, 0
)
units_sample["AGE_50_64"] = np.where(
    (units_sample["MEAN_AGE"] >= 50) & (units_sample["MEAN_AGE"] < 65), 1, 0
)

# create_estdata asserts this partition holds
assert (
    units_sample[["AGE_UNDER_18", "AGE_18_34", "AGE_35_64", "AGE_OVER_65"]].sum(axis=1)
    == 1
).all(), "AGE_* brackets do not partition the units"

In [28]:
# foreign born = not a US citizen at birth. IPUMS CITIZEN: 0 born in the US,
# 1 born abroad of American parents (still native), 2 naturalized, 3+ not a citizen.
# NB: this used BPL_SEC, where BPL is birthplace and 1 = Alabama -- so "BPL >= 2" was
# true for 98.5% of units and FOREIGN_BORN was almost always 1.
units_sample["FOREIGN_BORN"] = np.where(
    (units_sample["CITIZEN_REF"] >= 2) | (units_sample["CITIZEN_SEC"] >= 2), 1, 0
)

In [29]:
units_sample["IN_COLLEGE"] = np.where(
    (units_sample["GRADEATT_REF"] >= 6) | (units_sample["GRADEATT_SEC"] >= 6), 1, 0
)

In [30]:
# this doesn't match the true MARST in df because some of the MARST people do not represent a decision unit
# this is true for most of these
units_sample["MARRIED"] = np.where(
    units_sample["MARST_REF"].isin([1, 2]) & units_sample["MARST_SEC"].isin([1, 2]),
    1,
    0,
)
# married unit and there are actually 2 people
units_sample["MARRIED_AND_TOGETHER"] = np.where(
    units_sample["MARRIED"] & (units_sample["PAIRED_UNIT"] == 1), 1, 0
)
# defined as either the reference or secondary person being divorced/widowed
units_sample["RECENTLY_WIDOWED_OR_DIVORCED"] = np.where(
    (units_sample["DIVINYR_REF"] == 2)
    | (units_sample["WIDINYR_REF"] == 2)
    | (units_sample["DIVINYR_SEC"] == 2)
    | (units_sample["WIDINYR_SEC"] == 2),
    1,
    0,
)
# referring to a recently married couple (or one of them, if they are living alone)
units_sample["RECENTLY_MARRIED"] = np.where(
    (units_sample["MARRINYR_REF"] == 2) & (units_sample["MARRINYR_SEC"] == 2), 1, 0
)

units_sample["MARRIED_MORE_THAN_YEAR"] = np.where(
    units_sample["MARRIED"] & ~units_sample["RECENTLY_MARRIED"], 1, 0
)

In [31]:
units_sample["IN_MILITARY_REF"] = np.where(units_sample["VETSTATD_REF"] == 12, 1, 0)
units_sample["IN_MILITARY_SEC"] = np.where(units_sample["VETSTATD_SEC"] == 12, 1, 0)
units_sample["IN_MILITARY"] = np.where(
    (units_sample["IN_MILITARY_REF"] == 1) | (units_sample["IN_MILITARY_SEC"] == 1),
    1,
    0,
)
units_sample["UNEMPLOYED"] = np.where(
    (units_sample["EMPSTAT_REF"] == 2) & (units_sample["EMPSTAT_SEC"] == 2), 1, 0
)
units_sample["IN_LABOR_FORCE"] = np.where(
    units_sample["EMPSTAT_REF"].isin([1, 2]) | units_sample["EMPSTAT_SEC"].isin([1, 2]),
    1,
    0,
)
units_sample["NOT_IN_LABOR_FORCE"] = np.where(units_sample["IN_LABOR_FORCE"] == 1, 0, 1)

In [32]:
RACE_GROUPS = {
    "WHITE": [1],
    "BLACK": [2],
    "INDIAN": [3],
    "AAPI": [4, 5, 6],
    "OTHER_RACE": [7, 8, 9],
}

for suffix in CATEGORIES:
    race = units_sample[f"RACE_{suffix}"]
    hisp = units_sample[f"HISPAN_{suffix}"]

    latino = hisp.ne(0) & hisp.ne(9)  # 9 = not reported, absent in 2018
    units_sample[f"LATINO_{suffix}"] = latino.astype(int)

    # Hispanic takes precedence, so the six categories stay mutually exclusive
    # and line up with the SE_B04001 area shares (non-Hispanic race + Hispanic).
    for name, codes in RACE_GROUPS.items():
        units_sample[f"{name}_{suffix}"] = (race.isin(codes) & ~latino).astype(int)

    units_sample[f"RACE_ETHNICITY_{suffix}"] = np.where(latino, 99, race)

In [33]:
units_sample.to_parquet(
    f"pums_{sample_size * 100:.0f}_{year}.parquet", compression="gzip"
)

In [34]:
# # Use SFRELATE if present, otherwise RELATE
# relate_col = "SFRELATE" if "SFRELATE" in df.columns else "RELATE"

# df["IS_SECONDARY"] = ((df["SFRELATE"] == 2) | (df["RELATE"] == 2)).astype(int)

# # Count per UNIT
# secondary_counts = (
#     df.groupby("UNIT", sort=False)["IS_SECONDARY"]
#     .sum()
#     .reset_index(name="N_SECONDARY")
# )
# print(secondary_counts["N_SECONDARY"].describe())
# print(((df["SFRELATE"] == 2) + (df["RELATE"] == 2)).max())

In [35]:
# people who've recently had children category
# NOTE: this is a little iffy since this only applies to the women

# def add_recent_child_flag(df: pd.DataFrame) -> pd.DataFrame:
#     """FER_CL cleaning and inference for whether a household recently had a child."""
#     df["FER_CL"] = df["FER"].fillna(0)
#     df["FER_CL"] = np.where(df["FER_CL"] == 2, 0, df["FER_CL"])
#     rec_child = df.groupby("SERIALNO")["FER_CL"].max()
#     df["REC_CHILD"] = rec_child.loc[df["SERIALNO"]].values
#     return df
# df = lclean.add_recent_child_flag(df)
# df["FER_CL"].value_counts()